# Part 6 — Conclusions and Evidence Synthesis

**BFT65B — Big Data Analytics in Finance | Student ID: 11230945**

Consolidate verified results for Q1–Q3 and specify limitations and local adaptation requirements. All calculations use saved evidence. No historical test rescoring or new model selection is performed.

## 1. Repository and runtime
Use a fresh CPU runtime and the pinned environment. The complete offline test suite requires Java 17 or later.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
java = shutil.which("java")
if java is None:
    raise RuntimeError("Java 17 or later is required")
os.environ["JAVA_HOME"] = str(Path(java).resolve().parents[1])
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
print(subprocess.run([java,"-version"],check=True,capture_output=True,text=True).stderr)
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"requirements.txt")],check=True)
sys.path.insert(0,str(REPO/'src'))

## 2. Mandatory offline validation
A failure or skipped test stops execution. Tests use synthetic fixtures; they do not access the historical test workspace.

In [ ]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path('/content/part6_tests.xml')
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable,'-m','pytest','-q','-ra','--tb=short','--show-capture=no',f'--junitxml={result_path}','tests'],cwd=REPO,env=dict(os.environ,PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',SPARK_LOCAL_IP='127.0.0.1',PYSPARK_PYTHON=sys.executable,MPLBACKEND='Agg'),capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if not result_path.is_file(): raise RuntimeError('Missing pytest report')
cases=list(ET.parse(result_path).getroot().iter('testcase'))
problems=[(case,node) for case in cases for node in case if node.tag in ('failure','error','skipped')]
for case,node in problems:
    print(case.get('classname'),case.get('name'),node.tag,node.get('message'),node.text or '')
if result.returncode or not cases or problems: raise RuntimeError('Offline validation failed; inspect diagnostics')
TESTS_PASSED=True
print(f'Mandatory offline validation: PASS ({len(cases)} tests, zero skips)')

## 3. Verified evidence chain and report-front-page quantities
Resolve all five active pointers and the completed modelling run. Check hashes, ancestry and aggregate reconciliations without decoding model inputs or scoring estimators. The full verification results remain in Notebooks 01–07.

In [ ]:
from google.colab import drive
from fdic_ews.conclusions import collect
import pandas as pd
import numpy as np
from IPython.display import display, HTML, Markdown
from html import escape

drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/11230945')
assert TESTS_PASSED
E = collect(WORKSPACE)
assert E['status'] == 'PASS'
M = E['modelling']
SELECTED = M['selected_model']

PRIMARY = '#365C72'

def table(data, title, formats=None):
    df = pd.DataFrame(data).copy()
    styler = (df.style.hide(axis='index').format(formats or {}, na_rep='—', escape='html')
              .set_table_styles([
                  {'selector':'th','props':[('background-color',PRIMARY),('color','white'),('padding','8px 10px'),('font-size','11px')]},
                  {'selector':'td','props':[('padding','7px 10px'),('color','#243746'),('border-bottom','1px solid #E2E6EA'),('font-size','11px')]},
                  {'selector':'tbody tr:nth-child(odd)','props':[('background-color','#F2F5F7')]},
                  {'selector':'tbody tr:nth-child(even)','props':[('background-color','white')]}]))
    display(HTML(f'<div style="margin:24px 0 30px;overflow-x:auto"><p><strong>{escape(title)}</strong></p>{styler.to_html()}</div>'))

source=E['source']
front=[('Acquisition method','FDIC BankFind API, quarter-by-quarter'),
       ('Acquisition started (UTC)',source['created_utc']),('Acquisition sealed (UTC)',source['sealed_utc']),
       ('Financial quarters',str(source['quarters'])),
       *[(k.title()+' rows',f'{v:,}') for k,v in source['row_counts'].items()],
       ('Distinct financial CERTs',f"{source['distinct_certs']:,}"),
       ('Development decision periods','1984Q1–2000Q4; 2002Q1–2006Q4'),
       ('Labelled development rows',f"{M['development_rows']:,}"),
       ('Distinct represented development events',f"{M['development_events']:,}"),
       ('Historical test decisions','2008Q1–2014Q4'),
       ('Labelled test rows',f"{int(M['test_metrics'][0]['rows']):,}"),
       ('Represented failures in test calendar',f"{E['features']['summary']['test_distinct_failure_events_in_calendar_period']:,}"),
       ('Represented failures through test horizons',f"{E['features']['summary']['test_distinct_positive_events_through_final_horizon']:,}")]
table(pd.DataFrame(front,columns=['Quantity','Value']), 'Table 1. Study scope and report-front-page evidence')
table([{'Stage':k.title(),'SHA-256':v} for k,v in E['lineage'].items()], 'Table 2. Verified evidence identities')
print('Evidence synthesis:',E['status'])
print('Verification scope:',E['verification_scope'])

## 4. Q1 — Storage, processing and distributed computing
Compare equivalent workloads; speedups above one indicate lower median latency than the reference. File candidate counts are not physical bytes read. Report measured memory alongside latency, and do not infer a multi-node break-even from local Spark or synthetic scale-up.

In [ ]:
S=E['storage']; P=E['processing']
table(S['sizes'],'Table 3. Storage footprint relative to CSV',{'mib':'{:,.2f}','reduction_pct':'{:.2f}'})
table(S['queries'],'Table 4. Storage workload comparison',{'seconds':'{:.4f}','speedup_vs_csv':'{:.2f}','peak_rss_mib':'{:,.1f}','candidate_fragments':'{:.0f}'})
proc=pd.DataFrame(P['results'])
table(proc[proc.scale.eq(1)],'Table 5. Original-panel processing performance',{'input_rows':'{:,.0f}','seconds':'{:.4f}','speedup_vs_pandas':'{:.2f}','time_ratio_vs_1x':'{:.2f}','peak_rss_mib':'{:,.1f}'})
table(proc[proc.scale.gt(1)],'Table 6. Synthetic scale-up performance',{'input_rows':'{:,.0f}','seconds':'{:.4f}','speedup_vs_pandas':'{:.2f}','time_ratio_vs_1x':'{:.2f}','peak_rss_mib':'{:,.1f}'})
ram=float(P['environment']['ram_mib'])
local=proc[proc.engine.eq('polars')].copy()
local['rss_to_host_ram_pct']=100*local.peak_rss_mib/ram
local=local[['workload','scale','seconds','peak_rss_mib','rss_to_host_ram_pct']]
table(local,'Table 7. Polars observed memory relative to recorded host RAM',{'seconds':'{:.4f}','peak_rss_mib':'{:,.1f}','rss_to_host_ram_pct':'{:.2f}'})
parallel=pd.DataFrame(P['parallelism'])
for col in parallel.columns.difference(['workload']): parallel[col]=pd.to_numeric(parallel[col])
table(parallel,'Table 8. One- versus two-thread Polars diagnostics',{c:'{:.4f}' for c in parallel.columns if c!='workload'})
print(f"Measured storage trials: {S['measured_trials']:,}; processing trials: {P['measured_trials']:,}; recorded host RAM: {ram:,.1f} MiB")
print('Smallest measured storage representation:',min(S['sizes'],key=lambda r:r['mib'])['format'])
for w,g in proc[proc.scale.eq(1)].groupby('workload'):
    winner=g.loc[g.seconds.idxmin()]
    print(f"{w}: lowest observed median latency = {winner.engine} ({winner.seconds:.4f} s)")

**Interpretation boundary.** Use the measured workload winners and the full latency/size trade-offs rather than assuming a single format is uniformly best. Host RAM is context, not an available-memory or out-of-memory guarantee. Benchmark memory definitions and timing boundaries differ between Parts 1 and 3. The original Bronze layout remains immutable. A cluster requires evidence on service deadlines, concurrent workloads, data locality, memory pressure and network/provisioning cost; actual multi-node break-even has not been measured.

## 5. Q2 — Data quality and temporal integrity
Separate source-report problems from outcome adjudication. Count flagged observations, not unique institutions. Flags may overlap. Source availability sensitivity is not a model sensitivity experiment.

In [ ]:
Q=E['quality']; G=E['features']; ss=Q['summary']
table([{'Diagnostic':k,'Value':f'{ss[k]:,}'} for k in ['source_rows','panel_rows','rows_removed','asset_to_null','gap_rows','event_ledger_rows','distinct_certs']], 'Table 9. Silver accounting')
fields=pd.DataFrame(Q['fields'])
for c in ['missing_pct','zero_pct']: fields[c]=pd.to_numeric(fields[c])
table(fields[['field','missing_pct','zero_pct']].sort_values('missing_pct',ascending=False).head(12),'Table 10. Highest field-level missingness',{'missing_pct':'{:.2f}','zero_pct':'{:.2f}'})
table(Q['flags'],'Table 11. Quality flags',{'flagged_rows':'{:,.0f}'})
table(Q['terminal'],'Table 12. Retrospective terminal audit (not a modelling filter)')
events=ss['event_diagnostics']
table([{'Diagnostic':k,'Records or events':v} for k,v in events.items() if isinstance(v,int)],'Table 13. Event linkage and classification evidence',{'Records or events':'{:,.0f}'})
a=pd.DataFrame(Q['availability'])
a['quarter_offset']=(pd.PeriodIndex(a.decision_quarter,freq='Q').asi8-pd.PeriodIndex(a.report_quarter,freq='Q').asi8)
for c in ['lag_days','available_rows','report_rows']: a[c]=pd.to_numeric(a[c])
a=a.groupby(['lag_days','quarter_offset'],as_index=False)[['available_rows','report_rows']].sum()
a['available_pct']=100*a.available_rows/a.report_rows
table(a,'Table 14. Source-report availability by assumed lag and decision-quarter offset',{'available_rows':'{:,.0f}','report_rows':'{:,.0f}','available_pct':'{:.2f}'})
sp=pd.DataFrame(G['splits'])
for c in ['risk_rows','labelled','positive','negative','censored']: sp[c]=pd.to_numeric(sp[c])
sp['censored_pct']=100*sp.censored/sp.risk_rows.replace(0,np.nan)
table(sp[['split','risk_rows','labelled','positive','negative','censored','censored_pct']],'Table 15. Gold risk and outcome accounting',{**{c:'{:,.0f}' for c in ['risk_rows','labelled','positive','negative','censored']},'censored_pct':'{:.2f}'})
print('Horizon-invariance diagnostic:',ss['horizon_case'])
print('Gold information-availability policy:',G['policy'])

**Interpretation boundary.** No future terminal class selects historical observations. Failures and assistance are distinct; an uncorroborated end is not a confirmed merger. Reported capital zeros and conditional YTD income conversions remain qualified. The assumed reporting lag and retrospective FDIC snapshot do not prove true historical availability. The labelled population excludes unresolved outcomes, which may induce selection effects.

## 6. Q3 — Historical early-warning feasibility
The model and economic threshold remain frozen. Ranking, operating performance and distinct event detection answer different questions. The calendar population is smaller than the full forecast-horizon population. Annual event counts cannot be added.

In [ ]:
metrics=pd.DataFrame(M['test_metrics']); calendar=pd.DataFrame(M['calendar'])
cols=['rows','positives','prevalence','ap','roc_auc','brier','log_loss','recall_top5pct_by_quarter','precision','recall','fpr','alert_rate','threshold','alerts','fp','events','detected_events','missed_events','event_recall','median_lead_days','lead_at_least_90_days','loss_usd_thousands']
for c in cols: metrics[c]=pd.to_numeric(metrics[c],errors='coerce')
for c in ['represented_events','distinct_certs','detected_events','event_recall','median_lead_days']: calendar[c]=pd.to_numeric(calendar[c],errors='coerce')
pct={c:'{:.2%}' for c in ['prevalence','recall_top5pct_by_quarter','precision','recall','fpr','alert_rate','event_recall']}
rank=['model','rows','positives','prevalence','ap','roc_auc','brier','log_loss','recall_top5pct_by_quarter']
table(metrics[rank],'Table 16. Historical test discrimination and probability error',{**pct,**{c:'{:.6f}' for c in ['ap','roc_auc','brier','log_loss']},'rows':'{:,.0f}','positives':'{:,.0f}'})
table(metrics[['model','threshold','alerts','fp','precision','recall','fpr','alert_rate']],'Table 17. Frozen operating policy',{**pct,'threshold':'{:.9f}','alerts':'{:,.0f}','fp':'{:,.0f}'})
table(metrics[['model','events','detected_events','missed_events','event_recall','median_lead_days','lead_at_least_90_days']],'Table 18. Failure-event detection through the test horizons',{**pct,'median_lead_days':'{:.1f}'})
table(calendar,'Table 19. Represented failures within the test calendar',pct)
refs=pd.DataFrame(M['reference_policies'])[['policy','fp','missed_events','loss_usd_thousands']].rename(columns={'policy':'Policy'})
sel=metrics[metrics.model.eq(SELECTED)][['model','fp','missed_events','loss_usd_thousands']].rename(columns={'model':'Policy'})
loss=pd.concat([sel,refs],ignore_index=True)
for c in ['fp','missed_events','loss_usd_thousands']: loss[c]=pd.to_numeric(loss[c])
loss['loss_usd_million']=loss.pop('loss_usd_thousands')/1000
table(loss,'Table 20. Economic scenario comparison (nominal USD millions)',{'fp':'{:,.0f}','missed_events':'{:,.0f}','loss_usd_million':'{:,.3f}'})
table([{'Diagnostic':k,'Value':v} for k,v in M['derived'].items()],'Table 21. Derived research indicators',{'Value':'{:.4f}'})
annual=pd.DataFrame(M['annual']); annual=annual[annual.model.eq(SELECTED)].copy()
for c in ['year','prevalence','ap','roc_auc','recall','event_recall']: annual[c]=pd.to_numeric(annual[c],errors='coerce')
table(annual[['year','prevalence','ap','roc_auc','recall','event_recall']],'Table 22. Annual decision-period stability',{**pct,'ap':'{:.4f}','roc_auc':'{:.4f}','year':'{:.0f}'})
weak=annual.loc[annual.event_recall.idxmin()]
print(f"Lowest annual event recall: {int(weak.year)} ({weak.event_recall:.2%}); AP in that year: {weak.ap:.4f}")
print('Selected development model:',SELECTED,'| mean fold AP:',M['selected_mean_ap'])
print('Frozen thresholds:',M['thresholds'])
print('Economic cost source:',M['cost']['missed_cost_source'])
print('Eligible dated FDIC cost events:',M['cost']['eligible_dated_cost_events'])
print('Spark validation AP:',M['spark']['ap'],'| validation sampled:',M['spark']['validation_sampled'])

**Interpretation boundary.** Top-5% recall is a positive-row ranking measure on labelled observations. Failure-event detection is not causal prevention; lead time depends on assumed decision-time availability. The economic scenario is based on frozen cost assumptions, not realized savings. Low annual operating recall can coexist with strong discrimination. The historical holdout is retrospective and not independently blind; post-test diagnostics do not change the locked policy.

## 7. Research limitations and adaptation to Vietnam
The following matrix defines requirements for a future local pilot, not empirical findings about Vietnamese banks. No US score or threshold is transferred. See `docs/part6_protocol.md` for sources and interpretation boundaries.

In [ ]:
limitations=[
('Information availability','Assumed publication lag; historical revisions unobserved','Versioned submissions with actual release and revision timestamps'),
('Event and accounting semantics','Unlinked events, uncertain exits, ambiguous capital zeros and historical income conventions','Local reporting dictionary and dated, independently adjudicated events'),
('Evaluation population','Censored rows; repeated decision windows; retrospective crisis-period holdout','Quantify censoring and event-level uncertainty; independently held-out or prospective validation'),
('Economic decision','Asset-scaled proxy costs; fixed threshold may transfer poorly','Estimate review capacity and local costs; lock a policy on local development'),
('Scale','Single-node timings and synthetic replication; no multi-node break-even','Measure concurrent workloads, memory, deadlines, network and cost before cluster adoption'),
('Governance','Research demonstration; no operational deployment claim','Authorized access, human review, audit trails and ongoing drift monitoring')]
table(pd.DataFrame(limitations,columns=['Domain','Observed limitation','Required next evidence']),'Table 23. Limitations and local adaptation requirements')
print('Part 6 synthesis: PASS')
print('Historical test evidence remains closed; no models scored.')
print('Development seal:',E['lineage']['development'])
print('Test seal:',E['lineage']['test'])

### References and report handoff
- BFT65B examination instructions, Parts 1–5 protocols and executed evidence.
- [BCBS risk-data aggregation principles](https://www.bis.org/publications/201301-guidelines-principles-effective-risk-data-aggregation-and-risk-reporting) and [Core Principles for effective banking supervision](https://www.bis.org/committees/bcbs/basel-framework/standard/bcp?allChapters=true).
- [Vietnam Law on Credit Institutions No. 32/2024/QH15](https://vanban.chinhphu.vn/?classid=1&docid=211190&pageid=27160&typegroupid=3) and [amending Law No. 96/2025/QH15](https://vanban.chinhphu.vn/?classid=1&docid=214588&pageid=27160&typegroupid=3); confirm subsequent amendments and applicable rules before local deployment.

Save the executed notebook with outputs. Use the evidence to answer Q1–Q3 in the report and identify limitations and local requirements. Do not commit raw data or predictions.